# DDPM: from noise to samples

**Denoising Diffusion Probabilistic Models** · Ho, Jain & Abbeel · 2020

[Original paper](https://arxiv.org/abs/2006.11239)

**Learning goal:** Implement forward noising, an epsilon predictor, and a complete tiny reverse chain on one-dimensional data.

## Before starting

Gaussian noise, MSE, optimization, and introductory world models.

Allow roughly 40–60 minutes including exercises. CPU, PyTorch, and Matplotlib; no model or dataset downloads. This is a study-time estimate, not a runtime benchmark.

## Problem and idea

Diffusion gradually corrupts data and learns to reverse that process. The model receives x_t and t and predicts added noise. At inference, true x0 is unavailable: generation starts from Gaussian noise.

This mechanism helps explain diffusion world models but does not itself condition on actions or environment time. Each sample here is one scalar from a two-Gaussian mixture, a small formula-checking task rather than image generation or DIAMOND.

## Reproduction boundary

One-dimensional DDPM with a tiny MLP and short schedule; no U-Net, image benchmarks, classifier-free guidance, or world-model rollout.

The code is an original educational implementation of mathematical ideas from the paper, not a copy of an author repository. Passing assertions verifies the stated implementation properties, not the paper’s results.

## Formula → tensors

`alpha_t = 1-beta_t`; `abar_t = product(alpha_0...alpha_t)`. `x_t = sqrt(abar_t)*x0 + sqrt(1-abar_t)*epsilon`. Loss is `MSE(epsilon_theta(x_t,t), epsilon)`.

Reverse mean is `(x_t - beta_t/sqrt(1-abar_t)*eps_hat)/sqrt(alpha_t)`. We use fixed posterior variance `beta_t*(1-abar_prev)/(1-abar_t)`. Add no noise at final t=0. Diffusion time and environment rollout time are distinct axes.

In [ ]:
%matplotlib inline
import math
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| CPU | seed=42')

## Step 1

Check the schedule and recover x0 using known noise. This is an oracle sanity check, not generation.

In [ ]:
T = 80
beta = torch.linspace(1e-4, 0.2, T)
alpha = 1 - beta; abar = alpha.cumprod(0)
previous = torch.cat([torch.ones(1), abar[:-1]])
posterior_variance = beta * (1 - previous) / (1 - abar)
assert (abar[1:] < abar[:-1]).all() and posterior_variance[0] == 0
x0 = torch.randn(128, 1); epsilon = torch.randn_like(x0); t = torch.randint(T, (128,))
xt = abar[t, None].sqrt() * x0 + (1 - abar[t, None]).sqrt() * epsilon
recovered = (xt - (1 - abar[t, None]).sqrt() * epsilon) / abar[t, None].sqrt()
assert torch.allclose(recovered, x0, atol=1e-4)
print('terminal alpha_bar:', abar[-1].item())

## Step 2

Train a tiny MLP to predict epsilon, resampling training data each step.

In [ ]:
def data(batch):
    return (torch.randint(0, 2, (batch, 1)).float() * 2 - 1) * 2 + 0.3 * torch.randn(batch, 1)
predictor = nn.Sequential(nn.Linear(2, 64), nn.SiLU(), nn.Linear(64, 64), nn.SiLU(), nn.Linear(64, 1))
def predict(x, time):
    return predictor(torch.cat([x, time[:, None].float() / (T - 1)], -1))
optimizer = torch.optim.Adam(predictor.parameters(), lr=0.003)
history = []
for step in range(400):
    clean = data(128); time = torch.randint(T, (128,)); noise = torch.randn_like(clean)
    noisy = abar[time, None].sqrt() * clean + (1 - abar[time, None]).sqrt() * noise
    loss = F.mse_loss(predict(noisy, time), noise)
    optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step(); history.append(loss.item())
assert math.isfinite(history[-1])
plt.plot(history, color='#d73535', alpha=0.6); plt.xlabel('step'); plt.ylabel('epsilon MSE'); plt.show()

## Step 3

The reverse chain never sees clean data. Histograms are diagnostic: this small network may generate poor samples.

In [ ]:
with torch.no_grad():
    samples = torch.randn(1000, 1)
    for index in reversed(range(T)):
        time = torch.full((len(samples),), index, dtype=torch.long)
        epsilon_hat = predict(samples, time)
        mean = (samples - beta[index] / (1 - abar[index]).sqrt() * epsilon_hat) / alpha[index].sqrt()
        samples = mean + posterior_variance[index].sqrt() * torch.randn_like(samples) if index else mean
assert torch.isfinite(samples).all()
plt.hist(data(1000).flatten(), bins=50, density=True, alpha=0.4, color='gray', label='data')
plt.hist(samples.flatten(), bins=50, density=True, alpha=0.5, color='#d73535', label='generated')
plt.legend(); plt.xlabel('scalar value'); plt.show()

## Reading the result

Training loss averages different noise levels; inspect MSE by t separately. Schedule indexing or noise added at t=0 can break sampling despite valid backward. Matching means does not establish matching distributions.

## Independent experiments

1. Remove t from the predictor.
2. Make terminal abar near 1: is Gaussian initialization justified?
3. Add action conditioning and specify the required transition dataset.

Write a prediction first, then change one factor at a time. Reset the kernel and rerun all cells for a controlled comparison. Save a copy for your changes.

## Reasoning check — after experimenting

Without t, the network mixes denoising tasks. If terminal samples retain strong data signal, normal initialization mismatches the forward endpoint. Action-conditioned world models need observation/action/next-observation triples and held-out episodes.

## What to record in your lab notes

- Hypothesis and changed parameter.
- Seed, data size, step budget, and precision.
- Before/after diagnostics, plot, and explanation of unexpected results.
- What this experiment **does not** establish.

Use “Open Studio workspace” above the lesson to continue in the related tools.